# 🚀 Space Cloud V7: Earth Training Facility (Colab Runner)

**Mission Objective:** 
This notebook acts strictly as a remote execution terminal. It utilizes cloud GPU instances (like Google Colab T4/A100) to train the heavy Time-Series CNN on the 50GB WSTS dataset. 

**Architectural Constraint:**
To maintain the strict separation between Earth training and Space inference, **no Python logic is written in this notebook**. It solely executes the tracked `.py` scripts from the Git repository.

### Step 0: Hardware Verification
Before pulling the massive dataset, we must ensure the runtime is connected to a GPU. Training on a CPU will take weeks.

In [ ]:
!nvidia-smi

### Step 1: Environment Setup
Install the required libraries. We specifically need `h5py` for lazy-loading the dataset and `onnx` for the final model export.

In [ ]:
!pip install torch torchvision h5py onnx numpy onnxruntime pandas

### Step 2: Clone the Thesis Repository
Pull the exact codebase from GitHub to ensure the execution matches the local environment. 

*Instructions: Replace `YOUR_USERNAME` with your actual GitHub username.*

In [ ]:
!git clone https://github.com/YOUR_USERNAME/Thesis-Polimi-Guazzi.git

# Change directory directly into the Earth Training Facility
%cd /content/Thesis-Polimi-Guazzi/Version\ 7/1_earth_training_facility/
!ls -la

### Step 3: The NVMe Data Hack (Critical for I/O Performance)
The WSTS dataset is hosted on Google Drive. However, reading a 50GB HDF5 file directly over the Google Drive network mount causes a massive I/O bottleneck. Here, we mount the drive and copy the dataset directly to the Colab machine's local high-speed NVMe storage.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Note: Update this path if you named your Google Drive folder differently
!cp /content/drive/MyDrive/Wildfire_Project/wsts.hdf5 ./wsts.hdf5
!ls -lh wsts.hdf5

### Step 4: Model Training
Execute the PyTorch training loop. The script uses lazy-loading (`__getitem__`) to stream the HDF5 data into RAM batch-by-batch, preventing OOM crashes.

In [ ]:
# Export the environment variable so the script knows where the local NVMe file is
!export WSTS_DATA_PATH='./wsts.hdf5' && python src/train_resnet.py

### Step 5: Freeze and Export to ONNX
This step mathematically freezes the PyTorch weights, bakes the dataset's Normalization parameters (Mean/Std) into the first layer, and exports a stateless `.onnx` graph for the orbital payload.

In [ ]:
!python src/export_to_onnx.py
!ls -lh checkpoints/

### Step 6: Artifact Retrieval
Copy the tiny, flight-ready payload model back to Google Drive (or download it directly) so it can be dragged into the `2_orbit_payload` folder on your local machine.

In [ ]:
# Push it back to Drive for safekeeping
!cp checkpoints/wsts_model.onnx /content/drive/MyDrive/Wildfire_Project/
print("✅ MISSION ACCOMPLISHED: ONNX model successfully transferred to cloud storage.")

# Alternatively, trigger a direct browser download
from google.colab import files
files.download('checkpoints/wsts_model.onnx')